In [0]:
drivers = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/drivers.csv",
    header=True,
    inferSchema=True
)

print("Drivers table loaded successfully")

In [0]:
print("===== DRIVERS SCHEMA =====")
drivers.printSchema()

In [0]:
customers = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/customers.csv",
    header=True,
    inferSchema=True
)

delivery_events = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/delivery_events.csv",
    header=True,
    inferSchema=True
)

driver_monthly_metrics = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/driver_monthly_metrics.csv",
    header=True,
    inferSchema=True
)

facilities = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/facilities.csv",
    header=True,
    inferSchema=True
)

fuel_purchases = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/fuel_purchases.csv",
    header=True,
    inferSchema=True
)

loads = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/loads.csv",
    header=True,
    inferSchema=True
)

maintenance_records = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/maintenance_records.csv",
    header=True,
    inferSchema=True
)

routes = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/routes.csv",
    header=True,
    inferSchema=True
)

safety_incidents = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/safety_incidents.csv",
    header=True,
    inferSchema=True
)

trailers = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/trailers.csv",
    header=True,
    inferSchema=True
)

trips = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/trips.csv",
    header=True,
    inferSchema=True
)

trucks = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/trucks.csv",
    header=True,
    inferSchema=True
)

truck_utilization_metrics = spark.read.csv(
    "/Volumes/workspace/transportation_analytics/fleet_data/truck_utilization_metrics.csv",
    header=True,
    inferSchema=True
)

print("All 14 CSV files loaded successfully")

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.transportation_analytics
""")

print("Transportation analytics schema is ready")

In [0]:
from delta.tables import DeltaTable

bronze_table = "workspace.transportation_analytics.bronze_drivers"

if not spark.catalog.tableExists(bronze_table):
    (
        drivers.write
        .format("delta")
        .saveAsTable(bronze_table)
    )
    print("Bronze drivers table created")
else:
    (
        DeltaTable.forName(spark, bronze_table)
        .alias("target")
        .merge(
            drivers.alias("source"),
            "target.driver_id = source.driver_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Bronze drivers table updated using MERGE")

bronze_drivers = spark.table(bronze_table)

print("Row count:", bronze_drivers.count())
print("===== BRONZE DRIVERS SCHEMA =====")
bronze_drivers.printSchema()

In [0]:
print("===== BRONZE DRIVERS DATA =====")

bronze_drivers.show(10, truncate=False)

print("===== ROW COUNT =====")
print(bronze_drivers.count())

print("===== SCHEMA =====")
bronze_drivers.printSchema()

In [0]:
# ============================================================
# BRONZE LAYER - REMAINING TABLES
# ============================================================

bronze_tables = {
    "customers": (customers, "customer_id"),
    "delivery_events": (delivery_events, "event_id"),
    "driver_monthly_metrics": (driver_monthly_metrics, "driver_id, month"),
    "facilities": (facilities, "facility_id"),
    "fuel_purchases": (fuel_purchases, "purchase_id"),
    "loads": (loads, "load_id"),
    "maintenance_records": (maintenance_records, "maintenance_id"),
    "routes": (routes, "route_id"),
    "safety_incidents": (safety_incidents, "incident_id"),
    "trailers": (trailers, "trailer_id"),
    "trips": (trips, "trip_id"),
    "trucks": (trucks, "truck_id"),
    "truck_utilization_metrics": (truck_utilization_metrics, "truck_id, month")
}

for table_name, (df, merge_keys) in bronze_tables.items():

    full_table_name = f"workspace.transportation_analytics.bronze_{table_name}"

    print(f"\n========== {table_name.upper()} ==========")

    if not spark.catalog.tableExists(full_table_name):

        df.write \
            .format("delta") \
            .saveAsTable(full_table_name)

        print("Table created")

    else:

        merge_condition = " AND ".join(
            [
                f"target.{key.strip()} = source.{key.strip()}"
                for key in merge_keys.split(",")
            ]
        )

        (
            DeltaTable.forName(spark, full_table_name)
            .alias("target")
            .merge(
                df.alias("source"),
                merge_condition
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("Table updated using MERGE")

    bronze_df = spark.table(full_table_name)

    print("Row count:", bronze_df.count())
    print("Schema:")
    bronze_df.printSchema()

print("\n========================================")
print("ALL BRONZE TABLES PROCESSED SUCCESSFULLY")
print("========================================")

In [0]:
bronze_table_names = [
    "bronze_customers",
    "bronze_delivery_events",
    "bronze_drivers",
    "bronze_driver_monthly_metrics",
    "bronze_facilities",
    "bronze_fuel_purchases",
    "bronze_loads",
    "bronze_maintenance_records",
    "bronze_routes",
    "bronze_safety_incidents",
    "bronze_trailers",
    "bronze_trips",
    "bronze_trucks",
    "bronze_truck_utilization_metrics"
]

print("========================================")
print("       BRONZE LAYER VALIDATION")
print("========================================")

for table_name in bronze_table_names:

    full_name = f"workspace.transportation_analytics.{table_name}"

    print(f"\n{table_name}")

    if spark.catalog.tableExists(full_name):
        df = spark.table(full_name)

        print("Status    : OK")
        print("Row count :", df.count())
    else:
        print("Status    : MISSING")

print("\n========================================")
print("       BRONZE VALIDATION COMPLETE")
print("========================================")